In [1]:
import os
from pyspark.sql import SparkSession

# 1. Ép quyền root (Sửa lỗi Permission Denied)
os.environ["HADOOP_USER_NAME"] = "root"

# 2. Khởi tạo Spark với đầy đủ "cầu nối" Kafka và MySQL
spark = SparkSession.builder \
    .appName("Create_enrichDataset") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(">>> SparkSession đã sẵn sàng với quyền ROOT và các Driver kết nối!")

>>> SparkSession đã sẵn sàng với quyền ROOT và các Driver kết nối!


In [2]:
!pip install holidays

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 387.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 656.7 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 606.7 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.2
    Uninstalling python-dateutil-2.8.2:
      Successfully uninstalled python-dateutil-2.8.2


In [6]:
import pandas as pd
import os
from pyspark.sql import SparkSession, functions as F

# 1. Đọc file CSV 
df_pd = pd.read_csv('holidays_optimized.csv')

# 2. Khởi tạo Spark với cấu hình Root
os.environ["HADOOP_USER_NAME"] = "root"
spark = SparkSession.builder.appName("Optimize_Holidays").getOrCreate()

# 3. Chuyển sang Spark DataFrame và TỐI ƯU
spark_holiday_df = spark.createDataFrame(df_pd) \
    .withColumn("DateKey", F.col("DateKey").cast("integer")) \
    .dropDuplicates(["DateKey"])

# 4. Ghi xuống HDFS dạng Parquet 
# Parquet giúp giữ nguyên định dạng Integer và Boolean (True/False)
path_hdfs = "hdfs://namenode:9000/data/bronze/holidays_optimized"
spark_holiday_df.write.mode("overwrite").parquet(path_hdfs)

print(f">>> TỐI ƯU THÀNH CÔNG: {spark_holiday_df.count()} ngày lễ đã được nạp lên HDFS!")
print("Cấu trúc bảng hiện tại:")
spark_holiday_df.printSchema()

>>> TỐI ƯU THÀNH CÔNG: 10 ngày lễ đã được nạp lên HDFS!
Cấu trúc bảng hiện tại:
root
 |-- DateKey: integer (nullable = true)
 |-- HolidayName: string (nullable = true)
 |-- IsHoliday: boolean (nullable = true)



In [4]:
import pandas as pd
import os
from pyspark.sql import SparkSession

# 1. Đọc file đã tối ưu (file bạn đang mở trong Excel)
weather_df = pd.read_csv('weather_optimized.csv')

# 2. Kiểm tra dữ liệu (để chắc chắn)
print("Các cột hiện có:", weather_df.columns.tolist())
print(f"Tổng số ngày: {len(weather_df)}")

os.environ["HADOOP_USER_NAME"] = "root"
spark = SparkSession.builder.appName("Upload_Weather").getOrCreate()

spark_weather_df = spark.createDataFrame(weather_df)

spark_weather_df.write.mode("overwrite").parquet("hdfs://namenode:9000/data/bronze/weather_optimized")

print("THÀNH CÔNG: Dữ liệu thời tiết đã nằm trên HDFS (Tầng Bronze)")

Các cột hiện có: ['DateKey', 'Temperature', 'Rainfall', 'Snowfall']
Tổng số ngày: 374
THÀNH CÔNG: Dữ liệu thời tiết đã nằm trên HDFS (Tầng Bronze)


In [5]:
from pyspark.sql import SparkSession
import os

# 1. Ép quyền root để không bị lỗi Permission Denied
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName("Retail_Enrich_Data_Ingestion") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,mysql:mysql-connector-java:8.0.33") \
    .getOrCreate()

print(">>> Đang nạp Weather...")
weather_df = spark.read.csv("weather_optimized.csv", header=True, inferSchema=True)
weather_df.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:9000/data/bronze/weather_optimized")
print("  --> [HDFS] Đã lưu Weather thành công.")

print(">>> Đang nạp Holiday...")
holiday_df = spark.read.csv("holidays_optimized.csv", header=True, inferSchema=True)
holiday_df.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:9000/data/bronze/holidays_optimized")
print("  --> [HDFS] Đã lưu Holiday thành công.")

print("\n HOÀN THÀNH: TẤT CẢ DỮ LIỆU BỔ TRỢ ĐÃ NẰM TRÊN HDFS BRONZE LAYER ")

>>> Đang nạp Weather...
  --> [HDFS] Đã lưu Weather thành công.
>>> Đang nạp Holiday...
  --> [HDFS] Đã lưu Holiday thành công.

 HOÀN THÀNH: TẤT CẢ DỮ LIỆU BỔ TRỢ ĐÃ NẰM TRÊN HDFS BRONZE LAYER 
